# SVD et ACP pour les réseaux radio

Ce carnet reproductible prolonge le carnet de réduction dimensionnelle de Jean-Pierre Morard avec trois usages radio : l’ACP d’une matrice de KPI, la séparation robuste comportement nominal/anomalies, et la SVD complexe d’un canal MIMO. Les données sont synthétiques, les graines sont fixées et les assertions servent de garde-fous numériques.

> **Limite.** Les valeurs générées illustrent les méthodes ; elles ne constituent ni une calibration RF ni une preuve de performance terrain.

In [ ]:
from importlib.metadata import version

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)
print({name: version(name) for name in ("numpy", "pandas", "matplotlib")})

## 1. De la SVD à l’ACP

Pour une matrice de KPI centrée $X$ contenant $n$ observations :

$$X = U\Sigma V^\top$$

- les colonnes de $V$ sont les directions de l’ACP, donc des combinaisons de KPI ;
- $U\Sigma$ contient les observations projetées sur ces directions ;
- $\sigma_i^2/(n-1)$ est la variance expliquée par la composante $i$.

Les KPI sont standardisés parce qu’ils n’ont pas la même unité. Une étude opérationnelle doit aussi séparer les régimes de trafic, environnements, technologies et bandes de fréquences.

In [ ]:
# Trente jours de mesures horaires et sept KPI corrélés.
n_samples = 30 * 24
hours = np.arange(n_samples)
latent = np.column_stack((
    np.sin(2 * np.pi * hours / 24 - np.pi / 2),
    np.sin(2 * np.pi * hours / (24 * 7)),
    rng.normal(size=n_samples),
))
kpis = ["RSRP_dBm", "RSRQ_dB", "SINR_dB", "throughput_Mbps", "PRB_pct", "BLER", "latency_ms"]
loadings = np.array([
    [0.20, 0.55, 0.10], [-0.25, 0.70, 0.10], [-0.30, 0.75, 0.15],
    [0.80, 0.20, 0.15], [0.90, 0.10, 0.10], [0.45, -0.45, 0.20],
    [0.55, -0.35, 0.25],
])
base = np.array([-95.0, -11.0, 12.0, 80.0, 65.0, 0.08, 15.0])
scale = np.array([6.0, 2.0, 4.0, 25.0, 20.0, 0.04, 8.0])
nominal = latent @ loadings.T + 0.08 * rng.normal(size=(n_samples, len(kpis)))
measurements = base + scale * nominal

# Incidents injectés uniquement pour tester un signal connu.
n_incidents = 24
incident_rows = np.sort(rng.choice(n_samples, size=n_incidents, replace=False))
measurements[incident_rows] += np.array([-18.0, -6.0, -12.0, -45.0, 20.0, 0.18, 35.0])
kpi_frame = pd.DataFrame(measurements, columns=kpis)
known_incident = np.zeros(n_samples, dtype=bool)
known_incident[incident_rows] = True

X = ((kpi_frame - kpi_frame.mean()) / kpi_frame.std(ddof=1)).to_numpy()
U, singular_values, Vh = np.linalg.svd(X, full_matrices=False)
scores = U * singular_values
explained_variance = singular_values**2 / (n_samples - 1)
explained_ratio = explained_variance / explained_variance.sum()
directions = Vh.T
rank = 2
X_rank = scores[:, :rank] @ Vh[:rank]
relative_error = np.linalg.norm(X - X_rank, ord="fro") / np.linalg.norm(X, ord="fro")

assert np.isclose(explained_ratio.sum(), 1.0)
assert np.allclose(X, scores @ Vh, atol=1e-12)

summary = pd.DataFrame({
    "singular_value": singular_values,
    "explained_variance_ratio": explained_ratio,
    "cumulative_ratio": np.cumsum(explained_ratio),
}, index=[f"PC{i}" for i in range(1, len(kpis) + 1)])
print(f"Erreur relative de la reconstruction de rang {rank}: {relative_error:.3f}")
display(summary.round(4))
display(pd.DataFrame(directions[:, :rank], index=kpis, columns=["PC1", "PC2"]).round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(summary.index, explained_ratio)
axes[0].plot(summary.index, np.cumsum(explained_ratio), color="tab:red", marker="o", label="cumul")
axes[0].set(title="Spectre de variance", ylabel="fraction expliquée", ylim=(0, 1.05))
axes[0].legend()
axes[1].scatter(scores[:, 0], scores[:, 1], c=np.where(known_incident, "tab:red", "tab:blue"), s=14, alpha=0.7)
axes[1].set(title="Deux premières composantes", xlabel="PC1", ylabel="PC2")
fig.tight_layout()
plt.show()

## 2. RPCA : faible rang plus anomalies parcimonieuses

Les KPI nominaux sont corrélés, tandis qu’une panne ou une interférence peut ne toucher que quelques cellules et instants. Le modèle bruité serait $X=L+S+E$. Ce TP exécute cependant la PCP exacte avec la contrainte $X=L+S$ : $E=X-L-S$ est seulement un résidu numérique, pas une estimation du bruit de mesure. Le bruit dense injecté doit être absorbé par $L$ ou $S$. Une PCP stable avec un budget de bruit demanderait une autre optimisation.

La séparation requiert des hypothèses d’identifiabilité : le faible rang ne doit pas être lui-même trop concentré, et un incident qui ressemble à un mode nominal peut être ambigu. Les étiquettes ne servent pas à l’ajustement. Le rappel au rang $k=24$ utilise le nombre d’incidents connu : c’est un classement, pas un détecteur calibré. La fraction de coefficients non nuls et la fausse alerte de la règle naïve « un coefficient non nul suffit » montrent pourquoi le support de $S$ ne désigne pas seulement des incidents.

In [ ]:
def soft_threshold(values, threshold):
    return np.sign(values) * np.maximum(np.abs(values) - threshold, 0.0)


def singular_value_threshold(matrix, threshold):
    left, values, right_h = np.linalg.svd(matrix, full_matrices=False)
    retained = values > threshold
    if not np.any(retained):
        return np.zeros_like(matrix)
    return (left[:, retained] * (values[retained] - threshold)) @ right_h[retained]


def robust_pca(matrix, tolerance=1e-7, max_iterations=1000):
    """Principal Component Pursuit par multiplicateurs de Lagrange augmentés."""
    matrix = np.asarray(matrix, dtype=float)
    rows, columns = matrix.shape
    penalty = 1.0 / np.sqrt(max(rows, columns))
    spectral_norm = np.linalg.norm(matrix, ord=2)
    multiplier = matrix / max(spectral_norm, np.max(np.abs(matrix)) / penalty)
    mu = 1.25 / spectral_norm
    mu_limit = mu * 1e7
    low_rank = np.zeros_like(matrix)
    sparse = np.zeros_like(matrix)
    matrix_norm = np.linalg.norm(matrix, ord="fro")

    for iteration in range(1, max_iterations + 1):
        low_rank = singular_value_threshold(matrix - sparse + multiplier / mu, 1.0 / mu)
        sparse = soft_threshold(matrix - low_rank + multiplier / mu, penalty / mu)
        residual = matrix - low_rank - sparse
        relative_residual = np.linalg.norm(residual, ord="fro") / matrix_norm
        if relative_residual < tolerance:
            break
        multiplier += mu * residual
        mu = min(mu * 1.5, mu_limit)
    else:
        raise RuntimeError("RPCA did not converge")
    return low_rank, sparse, iteration, relative_residual


low_rank, sparse, iterations, rpca_residual = robust_pca(X)
anomaly_score = np.linalg.norm(sparse, axis=1)
top_rows = np.argsort(anomaly_score)[-n_incidents:]
incident_recall_at_k = known_incident[top_rows].mean()
sparse_support = np.abs(sparse) > 1e-6
sparse_entry_fraction = sparse_support.mean()
naive_support_false_alarm_rate = sparse_support.any(axis=1)[~known_incident].mean()
assert rpca_residual < 1e-6
assert incident_recall_at_k >= 0.90
# Contre-exemple : le support non nul absorbe le bruit et ne localise pas les seuls incidents.
assert sparse_entry_fraction > 0.50
assert naive_support_false_alarm_rate > 0.50

display(pd.DataFrame({
    "RPCA iterations": [iterations],
    "relative residual": [rpca_residual],
    "estimated low-rank rank": [np.linalg.matrix_rank(low_rank, tol=1e-6)],
    "incident recall at k": [incident_recall_at_k],
    "nonzero S entry fraction": [sparse_entry_fraction],
    "naive nonzero-support false alarm rate": [naive_support_false_alarm_rate],
}).round(6))

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(hours, anomaly_score, linewidth=1.0, label="score RPCA")
ax.scatter(incident_rows, anomaly_score[incident_rows], color="tab:red", s=25, label="incident injecté", zorder=3)
ax.set(title="Résidu parcimonieux", xlabel="heure", ylabel=r"$\|S_t\|_2$")
ax.legend()
fig.tight_layout()
plt.show()

## 3. SVD complexe d’un canal MIMO

Pour $H=U\Sigma V^H$, $V$ fournit les directions de précodage, $U$ les directions de combinaison et $\Sigma$ la force des flux spatiaux. Sous les hypothèses du modèle, cette base diagonalise le canal en sous-canaux spatiaux.

In [ ]:
mimo_rng = np.random.default_rng(7)
n_receive = n_transmit = 4
H = (mimo_rng.normal(size=(n_receive, n_transmit)) + 1j * mimo_rng.normal(size=(n_receive, n_transmit))) / np.sqrt(2)
U_h, sigma_h, Vh_h = np.linalg.svd(H, full_matrices=False)
V_h = Vh_h.conj().T
H_reconstructed = (U_h * sigma_h) @ Vh_h
reconstruction_error = np.linalg.norm(H - H_reconstructed, ord="fro") / np.linalg.norm(H, ord="fro")
left_unitarity_error = np.linalg.norm(U_h.conj().T @ U_h - np.eye(n_receive))
right_unitarity_error = np.linalg.norm(V_h.conj().T @ V_h - np.eye(n_transmit))

snr_db = 10.0
snr_linear = 10 ** (snr_db / 10)
capacity_from_modes = np.sum(np.log2(1 + (snr_linear / n_transmit) * sigma_h**2))
_, log_det = np.linalg.slogdet(np.eye(n_receive) + (snr_linear / n_transmit) * H @ H.conj().T)
capacity_from_matrix = log_det / np.log(2)

assert reconstruction_error < 1e-12
assert left_unitarity_error < 1e-12
assert right_unitarity_error < 1e-12
assert np.isclose(capacity_from_modes, capacity_from_matrix)

display(pd.DataFrame({
    "spatial_mode": np.arange(1, len(sigma_h) + 1),
    "singular_value": sigma_h,
    "relative_mode_power": sigma_h**2 / np.sum(sigma_h**2),
}).round(4))
print(f"Rang numérique : {np.linalg.matrix_rank(H)}")
print(f"Capacité synthétique à {snr_db:.0f} dB : {capacity_from_modes:.3f} bit/s/Hz")
print(f"Erreur relative de reconstruction : {reconstruction_error:.2e}")

## 4. Guide de choix

| Données ou objectif | Variante recommandée |
|---|---|
| CSI et modes propres MIMO | SVD complexe tronquée |
| Très grands jeux de données réseau | SVD aléatoire |
| Télémétrie arrivant en continu | SVD incrémentale |
| Anomalies et pannes | SVD robuste / RPCA |
| Cellule × KPI × temps × fréquence | Décomposition tensorielle |

**Synthèse :** MIMO/CSI → SVD complexe ; KPI réseau → SVD/ACP robuste, ou SVD tronquée incrémentale.

Une troncature trop agressive peut supprimer les signatures rares recherchées. Le rang, les seuils et le prétraitement doivent être validés sur des régimes séparés et des incidents annotés.